# 在 Colab 中打开
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH02/ch02_HITL.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>



# 关于这个 Notebook

这个 Notebook 是一次 LangGraph Human-in-the-Loop（HITL，人类参与回路）模式的动手实践。它展示如何在保持代码紧凑、面向生产的同时，为 Agent 工作流加入精确的人类控制：从轻量审批门，到完整的交互式编辑。

## 你将学到什么

1. 如何接入 LangGraph interrupts（中断），暂停一次运行、收集人工输入，再以确定性的方式恢复执行。
2. 如何使用 `InMemorySaver` 对 state（状态）做 checkpoint（检查点），让运行可以暂停后继续，而不会丢失上下文。
3. 如何包装工具，使人工可以在工具执行前接受、编辑或覆盖一次调用。
4. 如何驱动简单的 ReAct 风格循环，并让工具调用真正经过人工审核。
5. 如何实现 parallel interrupts（并行中断），并通过一个 resume map（恢复映射）统一恢复。

## 模型与配置

Notebook 在启动时通过 provider switch（提供商开关）选择模型提供商。

* `LLM_PROVIDER=openai`：通过 `langchain_openai` 使用 OpenAI。
* `LLM_PROVIDER=openrouter`：通过 `ChatOpenAI` 配合自定义 `base_url` 使用 OpenRouter。

环境变量通过 `python-dotenv` 从 `.env` 加载：

* `LLM_PROVIDER`（`openai` 或 `openrouter`）
* 使用 OpenAI 时的 `OPENAI_API_KEY`
* 使用 OpenRouter 时的 `OPENROUTER_API_KEY`
* 可选的模型覆盖项：
  * `OPENAI_MODEL`（默认 `gpt-5.4-nano`）
  * `OPENROUTER_MODEL`（默认 `openai/gpt-5.4-nano`）

## 本 Notebook 展示的模式

**模式 A — 内容生成中的人工反馈循环（Human feedback loop）**
这是一个用于 LinkedIn 帖子的简单“写作—审核”循环。模型先起草，人类通过 `interrupt` 反复给出反馈，图会持续循环，直到人类输入 done。它适合任何能从迭代式修改中获益的短内容工作流。

**模式 B — 敏感调用前的 HITL checkpoint（人工检查点）**
模型先以 JSON 形式提出 HTTP 请求。在代码真正执行外部调用之前，人类可以批准、修改、要求更多上下文或拒绝。这是面向网络操作、金融操作等关键动作的一种极简但有效的 safety interlock（安全联锁）。

**模式 C — 审核并编辑 state（状态）**
模型生成一段简短摘要，人类可以直接修改文本，修改后的内容成为新的 state。这种模式非常适合合规检查或品牌语调审核。

**模式 D — 通过单个 resume map 恢复多个并行中断**
两个相互独立的 interrupt 同时触发。runner（运行器）会打印两份 payload（载荷），分别收集恢复值，然后一步恢复整个图。这可以作为多项并行审核任务的模板。

**模式 E — 微型 ReAct 循环中的工具调用审核**
工具通过 `add_hitl` 包装。执行前，人类可以接受调用、编辑参数，或直接返回一个 stub result（桩结果）。循环会持续到模型不再请求工具。这是 supervised tool use（受监督工具使用）的最小化示例。

**模式 F — 用于调试的静态中断（Static interrupts）**
在指定节点前后注册图级 interrupt，可以建立确定性的 breakpoint（断点）。这适合逐步调试和类似单元测试的验证。

## 交互式 runner 如何工作

* 每个 demo 都会构建一个 compiled graph（已编译图），并使用新的 thread id（线程 ID）启动，从而得到干净的状态。
* interrupt 发生时，终端会打印 payload 并等待输入。
* 你可以粘贴原始字符串或 JSON。对于模式 E 中的工具包装器，可以输入：

  * `{"type": "accept"}`
  * `{"type": "edit", "args": {"args": {"query": "weather in Zurich"}}}`
  * `{"type": "response", "args": "Skip for now"}`

对于 parallel interrupts，runner 会打印编号列表，为每个 interrupt id 收集一个恢复值，然后只通过一次 `Command(resume=...)` 统一恢复。

## 依赖项

* `langgraph`：图、interrupt、checkpoint 和 task
* `langchain_openai`：LLM 绑定
* `python-dotenv`：加载环境变量
* `requests`：模式 B 中用于真正执行 HTTP GET

## 如何运行

1. 创建 `.env` 文件，写入密钥和 provider 选择。将 `LLM_PROVIDER` 设置为 `openai` 或 `openrouter`，然后提供对应的 API Key（`OPENAI_API_KEY` 或 `OPENROUTER_API_KEY`）。
2. 运行 Notebook，并从 `main()` 打印的菜单中选择一个 demo。
3. 按照终端提示提供反馈或审批结果。

## 为什么这很重要

金融、研究和运营等真实系统往往同时需要 autonomy（自主性）与 control（控制）。这些模式展示了如何在不破坏 Agent 架构的前提下加入精确的人类控制。各模式可以自然组合，因此可以先从小范围开始，衡量效果，再在真正有价值的位置逐步增加更丰富的人工监督。


In [1]:
!pip install -q langgraph==0.6.7 langchain-openai==0.3.33 langchain==0.3.27 python-dotenv==1.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.5 MB/s eta 0:00:00


In [2]:
# --- Provider 与 API Key 配置 ---
# 方案 1（推荐）：在项目目录中创建 `.env` 文件，例如：
# LLM_PROVIDER=openai
# OPENAI_API_KEY=your_openai_key_here
# OPENAI_MODEL=gpt-4o-mini
#
# 或者：
# LLM_PROVIDER=openrouter
# OPENROUTER_API_KEY=your_openrouter_key_here
# OPENROUTER_MODEL=openai/gpt-4o-mini
#
# 方案 2：直接在 Notebook 中通过 `%env` 设置。

from dotenv import load_dotenv
import os

# 如果存在 `.env`，则从中加载。
load_dotenv()

PROVIDER = os.getenv("LLM_PROVIDER", "openai").strip().lower()
if PROVIDER not in {"openai", "openrouter"}:
    raise ValueError("LLM_PROVIDER must be 'openai' or 'openrouter'")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.4-nano")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Fallback（回退方案）：只询问当前所选 provider 对应的密钥。
if PROVIDER == "openai" and not OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY not found. Set it with `%env` or enter it below.")
    OPENAI_API_KEY = input("Enter your OPENAI_API_KEY: ").strip()

if PROVIDER == "openrouter" and not OPENROUTER_API_KEY:
    print("⚠️ OPENROUTER_API_KEY not found. Set it with `%env` or enter it below.")
    OPENROUTER_API_KEY = input("Enter your OPENROUTER_API_KEY: ").strip()

selected_model = OPENAI_MODEL if PROVIDER == "openai" else OPENROUTER_MODEL
print(f"✅ Provider: {PROVIDER} | Model: {selected_model}")



✅ Provider: openai | Model: gpt-5.4-nano


# 导入依赖

In [3]:
from __future__ import annotations
import os, uuid, json, re, sys
from typing import Any, Dict, List, Optional, TypedDict, Literal

from dotenv import load_dotenv

from langgraph.graph import StateGraph
from langgraph.constants import START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.graph.message import add_messages

from langchain_openai import ChatOpenAI
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool, BaseTool


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


# LLM 配置（通过 provider switch 在 OpenAI 与 OpenRouter 之间切换）

In [4]:
try:
    import langchain
    if not hasattr(langchain, "verbose"):
        langchain.verbose = False
    if not hasattr(langchain, "debug"):
        langchain.debug = False
    if not hasattr(langchain, "llm_cache"):
        langchain.llm_cache = None
except Exception:
    pass

if PROVIDER == "openrouter":
    LLM = ChatOpenAI(
        model=OPENROUTER_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        temperature=0,
    )
    print(f"Using OpenRouter model: {OPENROUTER_MODEL}")
else:
    LLM = ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
    print(f"Using OpenAI model: {OPENAI_MODEL}")

CHECKPOINTER = InMemorySaver()

def jdump(x):
    try:
        return json.dumps(x, indent=2, ensure_ascii=False, default=str)
    except Exception:
        return str(x)



Using OpenAI model: gpt-5.4-nano


# 模式 A：内容生成中的人工反馈循环

In [5]:
class AState(TypedDict, total=False):
    linkedin_topic: str
    generated_post: str
    human_feedback: List[str]

def a_model(state: AState) -> AState:
    topic = state["linkedin_topic"]
    fb = state.get("human_feedback", [])
    prompt = f"""
LinkedIn Topic: {topic}
Most recent human feedback: {fb[-1] if fb else "No feedback yet"}

Write a concise LinkedIn post. Consider feedback if present.
"""
    resp = LLM.invoke(prompt).content
    print("\n[model] Draft:\n" + resp + "\n")
    return {"generated_post": resp, "human_feedback": fb}

def a_human(state: AState):
    print("\n[human] awaiting feedback. Type done to finish")
    payload = {
        "generated_post": state["generated_post"],
        "message": "Provide feedback or type done"
    }
    feedback = interrupt(payload)
    print("[human] feedback:", feedback)
    if isinstance(feedback, str) and feedback.strip().lower() in {"done", "quit", "exit"}:
        return Command(goto="a_end", update={"human_feedback": state.get("human_feedback", []) + ["Finalised"]})
    return Command(goto="a_model", update={"human_feedback": state.get("human_feedback", []) + [str(feedback)]})

def a_end(state: AState) -> AState:
    print("\n[end] Final post:\n" + state["generated_post"])
    print("[end] Feedback trail:", state.get("human_feedback", []))
    return state

def build_graph_A():
    g = StateGraph(AState)
    g.add_node("a_model", a_model)
    g.add_node("a_human", a_human)
    g.add_node("a_end", a_end)
    g.set_entry_point("a_model")
    g.add_edge("a_model", "a_human")
    g.add_edge("a_end", END)
    return g.compile(checkpointer=CHECKPOINTER)


# 模式 B：敏感调用前的 HITL checkpoint（人工检查点）

In [6]:
class BState(TypedDict, total=False):
    proposed_request: Dict[str, Any]
    api_result: Dict[str, Any]
    decision: str
    human_note: str

def b_propose(state: BState) -> BState:
    prompt = "Return only JSON with keys url and params for GET to https://httpbin.org/get using q and limit."
    text = LLM.invoke(prompt).content
    m = re.search(r"\{.*\}", text, re.S)
    data = {"url": "https://httpbin.org/get", "params": {"q": "fallback", "limit": 1}}
    if m:
        try:
            data = json.loads(m.group(0))
        except Exception:
            pass
    return {"proposed_request": data}

def _merge_request(current: Dict[str, Any], update: Dict[str, Any]) -> Dict[str, Any]:
    new_req = dict(current)
    if "url" in update:
        new_req["url"] = update["url"]
    if isinstance(update.get("params"), dict):
        merged_params = dict(new_req.get("params") or {})
        merged_params.update(update["params"])
        new_req["params"] = merged_params
    return new_req

def b_gate(state: BState) -> Command[Literal["b_call", "b_gate", "b_rejected"]]:
    v = interrupt({
        "question": "Review sensitive request: approve, revise, ask for more human input, or reject",
        "proposed_request": state["proposed_request"],
        "schema": {
            "type": "object",
            "properties": {
                "action": {"enum": ["approve", "revise", "request_more_input", "reject"]},
                "update": {"type": "object"},
                "reason": {"type": "string"}
            },
            "required": ["action"]
        }
    })

    action = (v or {}).get("action")
    if action == "approve":
        return Command(goto="b_call", update={"decision": "approved"})

    if action == "reject":
        return Command(
            goto="b_rejected",
            update={"decision": "rejected", "human_note": str(v.get("reason") or "Rejected by human")}
        )

    if action == "request_more_input":
        extra = interrupt({
            "question": "Provide additional constraints before final approval",
            "proposed_request": state["proposed_request"],
            "schema": {
                "type": "object",
                "properties": {
                    "note": {"type": "string"},
                    "update": {"type": "object"}
                }
            }
        })
        upd = (extra or {}).get("update") or {}
        new_req = _merge_request(state["proposed_request"], upd)
        note = str((extra or {}).get("note") or "Additional human input captured")
        return Command(
            goto="b_gate",
            update={
                "proposed_request": new_req,
                "decision": "awaiting_final_approval",
                "human_note": note,
            },
        )

    upd = (v or {}).get("update") or {}
    new_req = _merge_request(state["proposed_request"], upd)
    return Command(goto="b_gate", update={"proposed_request": new_req, "decision": "revised"})

def b_call(state: BState) -> BState:
    import requests
    r = requests.get(state["proposed_request"]["url"], params=state["proposed_request"].get("params"), timeout=10)
    return {
        "api_result": {"status_code": r.status_code, "url": r.url},
        "decision": state.get("decision", "approved")
    }

def b_rejected(state: BState) -> BState:
    return {
        "api_result": {"status": "skipped", "reason": state.get("human_note", "Rejected by human")},
        "decision": "rejected",
    }

def build_graph_B():
    g = StateGraph(BState)
    g.add_node("b_propose", b_propose)
    g.add_node("b_gate", b_gate)
    g.add_node("b_call", b_call)
    g.add_node("b_rejected", b_rejected)
    g.set_entry_point("b_propose")
    g.add_edge("b_propose", "b_gate")
    g.add_edge("b_call", END)
    g.add_edge("b_rejected", END)
    return g.compile(checkpointer=CHECKPOINTER)

# 模式 C：审核并编辑 state（状态）

In [7]:
class CState(TypedDict, total=False):
    summary: str

def c_write(state: CState) -> CState:
    text = LLM.invoke("Write 2 sentences about why human in the loop matters for agents").content
    return {"summary": text}

def c_edit(state: CState) -> CState:
    res = interrupt({
        "task": "Edit the summary text",
        "summary": state["summary"],
        "schema": {
            "type": "object",
            "properties": {"edited_text": {"type": "string"}},
            "required": ["edited_text"]
        }
    })
    return {"summary": res["edited_text"]}

def build_graph_C():
    g = StateGraph(CState)
    g.add_node("c_write", c_write)
    g.add_node("c_edit", c_edit)
    g.set_entry_point("c_write")
    g.add_edge("c_write", "c_edit")
    g.add_edge("c_edit", END)
    return g.compile(checkpointer=CHECKPOINTER)

# 模式 D：通过单个 resume map 恢复并行中断

In [8]:
class DState(TypedDict, total=False):
    text_1: str
    text_2: str

def d_h1(state: DState):
    v = interrupt({"text_to_revise": state["text_1"]})
    return {"text_1": v}

def d_h2(state: DState):
    v = interrupt({"text_to_revise": state["text_2"]})
    return {"text_2": v}

def build_graph_D():
    g = StateGraph(DState)
    g.add_node("d_h1", d_h1)
    g.add_node("d_h2", d_h2)
    g.add_edge(START, "d_h1")
    g.add_edge(START, "d_h2")
    g.add_edge("d_h1", END)
    g.add_edge("d_h2", END)
    return g.compile(checkpointer=CHECKPOINTER)

# 模式 E：微型 ReAct 循环中的工具调用审核

In [9]:
def add_hitl(tool_obj: BaseTool | Any) -> BaseTool:
    if not isinstance(tool_obj, BaseTool):
        tool_obj = tool(tool_obj)

    @tool(tool_obj.name, description=tool_obj.description, args_schema=tool_obj.args_schema)
    def wrapped(**tool_input):
        request = [{
            "action_request": {"action": tool_obj.name, "args": tool_input},
            "config": {"allow_accept": True, "allow_edit": True, "allow_respond": True},
            "description": "Review this tool call"
        }]
        response = interrupt(request)[0]
        if response["type"] == "accept":
            return tool_obj.invoke(tool_input)
        if response["type"] == "edit":
            new_args = response["args"]["args"]
            return tool_obj.invoke(new_args)
        if response["type"] == "response":
            return response["args"]
        raise ValueError("Unsupported interrupt response type")
    return wrapped

@tool("echo_search")
def echo_search(query: str) -> str:
    """Demo tool that simulates a search by echoing the query."""
    return f"Search results for: {query}"

WRAPPED_SEARCH = add_hitl(echo_search)

@task
def e_call_model(messages: List[Dict[str, Any]]):
    return LLM.bind_tools([WRAPPED_SEARCH]).invoke(messages)

@task
def e_call_tool(tool_call: Dict[str, Any]) -> ToolMessage:
    obs = WRAPPED_SEARCH.invoke(tool_call["args"])
    return ToolMessage(content=obs, tool_call_id=tool_call["id"])

from langgraph.func import entrypoint as ep

@ep(checkpointer=CHECKPOINTER)
def e_agent(messages: List[Dict[str, Any]], previous: Optional[List[Dict[str, Any]]] = None):
    if previous is not None:
        messages = add_messages(previous, messages)
    llm_msg = e_call_model(messages).result()
    while True:
        tcs = getattr(llm_msg, "tool_calls", None) or []
        if not tcs:
            break
        tool_results = [e_call_tool(tc).result() for tc in tcs]
        messages = add_messages(messages, [llm_msg, *tool_results])
        llm_msg = e_call_model(messages).result()
    messages = add_messages(messages, llm_msg)
    return ep.final(value=llm_msg, save=messages)

# 模式 F：用于调试的静态中断

In [10]:
def build_graph_F():
    class S(TypedDict, total=False):
        x: int

    def a(state: S) -> S:
        return {"x": 1}

    def b(state: S) -> S:
        return {"x": state["x"] + 1}

    g = StateGraph(S)
    g.add_node("a", a)
    g.add_node("b", b)
    g.set_entry_point("a")
    g.add_edge("a", "b")
    g.add_edge("b", END)
    return g.compile(
        checkpointer=CHECKPOINTER,
        interrupt_before=["a"],
        interrupt_after=["b"]
    )



# 交互式 runner（运行器）

In [11]:
def wait_for_interrupt_and_prompt(app, cfg):
    """
    Drive interrupts from the terminal.
    Supports single payloads and lists used by wrapped tools.
    Also supports parallel interrupts by auto building a resume map.
    """
    state = app.get_state(cfg)
    ints = getattr(state, "interrupts", []) or []
    if not ints:
        print("No interrupts pending")
        return None

    if len(ints) > 1:
        print("\nMultiple interrupts pending:")
        for i, it in enumerate(ints, 1):
            print(f"[{i}] id={it.interrupt_id} value={jdump(it.value)}")
        print("Enter values per interrupt. Leave blank to echo original.")
        resume_map = {}
        for it in ints:
            val = input(f"Value for {it.interrupt_id}: ").strip()
            if val:
                # 优先尝试按 JSON 解析，否则保留为原始字符串。
                try:
                    resume_map[it.interrupt_id] = json.loads(val)
                except Exception:
                    resume_map[it.interrupt_id] = val
            else:
                resume_map[it.interrupt_id] = it.value
        return Command(resume=resume_map)

    # 单 interrupt（中断）路径。
    it = ints[0]
    print("\nInterrupt payload:")
    print(jdump(it.value))
    print("Enter resume value. Examples:")
    print("  Pattern B gate: {\"action\": \"approve\"}")
    print("  Pattern B revise: {\"action\": \"revise\", \"update\": {\"params\": {\"limit\": 3}}}")
    print("  Pattern B reject: {\"action\": \"reject\", \"reason\": \"Policy restriction\"}")
    print("  Tool wrapper accept: {\"type\": \"accept\"}")
    print("  Tool wrapper edit: {\"type\": \"edit\", \"args\": {\"args\": {\"query\": \"weather in NY\"}}}")
    print("  Tool wrapper response: {\"type\": \"response\", \"args\": \"Skip tool right now\"}")
    raw = input("resume> ").strip()
    if not raw:
        val = it.value
    else:
        try:
            val = json.loads(raw)
        except Exception:
            val = raw
    return Command(resume=val)

def run_pattern_A():
    app = build_graph_A()
    cfg = {"configurable": {"thread_id": f"A-{uuid.uuid4()}"}}
    topic = input("Enter LinkedIn topic: ").strip() or "Human in the loop for agents"
    stream = app.stream({"linkedin_topic": topic, "human_feedback": []}, config=cfg)
    while True:
        try:
            step = next(stream)
        except StopIteration:
            break
        if "__interrupt__" in step:
            cmd = wait_for_interrupt_and_prompt(app, cfg)
            stream = app.stream(cmd, cfg)
    print("Done A")

def run_pattern_B():
    app = build_graph_B()
    cfg = {"configurable": {"thread_id": f"B-{uuid.uuid4()}"}}
    _ = app.invoke({}, config=cfg)
    while True:
        cmd = wait_for_interrupt_and_prompt(app, cfg)
        _ = app.invoke(cmd, config=cfg)
        state = app.get_state(cfg)
        if not state.interrupts:
            print("Final state:", state.values)
            break
    print("Done B")

def run_pattern_C():
    app = build_graph_C()
    cfg = {"configurable": {"thread_id": f"C-{uuid.uuid4()}"}}
    _ = app.invoke({}, config=cfg)
    cmd = wait_for_interrupt_and_prompt(app, cfg)
    final = app.invoke(cmd, config=cfg)
    print("Final C:", final)

def run_pattern_D():
    app = build_graph_D()
    cfg = {"configurable": {"thread_id": f"D-{uuid.uuid4()}"}}
    _ = app.invoke({"text_1": "alpha", "text_2": "beta"}, config=cfg)
    cmd = wait_for_interrupt_and_prompt(app, cfg)
    final = app.invoke(cmd, config=cfg)
    print("Final D:", final)

def run_pattern_E():
    cfg = {"configurable": {"thread_id": f"E-{uuid.uuid4()}"}}
    user_msg = {"role": "user", "content": "Search for current weather in San Francisco"}
    stream = e_agent.stream([user_msg], cfg)
    while True:
        try:
            step = next(stream)
        except StopIteration:
            break
        if "__interrupt__" in step:
            cmd = wait_for_interrupt_and_prompt(e_agent, cfg)
            stream = e_agent.stream(cmd, cfg)
        else:
            print(step)
    print("Done E")

def run_pattern_F():
    app = build_graph_F()
    cfg = {"configurable": {"thread_id": f"F-{uuid.uuid4()}"}}
    _ = app.invoke({}, config=cfg)
    print("Breakpoint before a recorded. Resuming")
    _ = app.invoke(None, config=cfg)
    print("Breakpoint after b recorded. Resuming")
    final = app.invoke(None, config=cfg)
    print("Final F:", final)

def main():
    menu = """
Pick a demo
1. Human feedback loop for writing
2. HITL checkpoint before sensitive API call
3. Review and edit state
4. Parallel interrupts with resume map
5. Tool call review in a tiny ReAct loop
6. Static interrupts for debugging
q. Quit
> """
    while True:
        choice = input(menu).strip().lower()
        if choice == "1":
            run_pattern_A()
        elif choice == "2":
            run_pattern_B()
        elif choice == "3":
            run_pattern_C()
        elif choice == "4":
            run_pattern_D()
        elif choice == "5":
            run_pattern_E()
        elif choice == "6":
            run_pattern_F()
        elif choice in {"q", "quit", "exit"}:
            sys.exit(0)
        else:
            print("Unknown choice")

if __name__ == "__main__":
    main()



Pick a demo
1. Human feedback loop for writing
2. HITL checkpoint before sensitive API call
3. Review and edit state
4. Parallel interrupts with resume map
5. Tool call review in a tiny ReAct loop
6. Static interrupts for debugging
q. Quit
> 1
Enter LinkedIn topic: AI Agents

[model] Draft:
AI agents are shifting the conversation from “using AI” to **delegating work to AI**.

Instead of a single chatbot reply, agents can:
- plan steps toward a goal  
- use tools (email, calendars, docs, code)  
- take actions and learn from results  

The upside: faster execution.  
The risk: confident automation without guardrails.

What I’m watching right now: **agent reliability** (permissions, evaluation, monitoring) more than flashiness.

Curious—where do you think AI agents will create the biggest impact first: support, sales, engineering, or operations?


[human] awaiting feedback. Type done to finish

Interrupt payload:
{
  "generated_post": "AI agents are shifting the conversation from “using

SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
